In [1]:
import speech_recognition as sr
from gtts import gTTS
import playsound
import os
from googletrans import Translator
from transformers import T5Tokenizer, T5ForConditionalGeneration
from torch.utils.data import Dataset, DataLoader
import torch
import pandas as pd
from torch.utils.data import Dataset

/Users/siddhantanandjadhav/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/siddhantanandjadhav/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:

class TranslationDataset(Dataset):
    def __init__(self, filepath, tokenizer, source_lang, target_lang, max_length=128):
        self.data = pd.read_csv(filepath)
        self.tokenizer = tokenizer
        self.source_lang = source_lang
        self.target_lang = target_lang
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        input_text = self.data.iloc[idx]["Input Text"]
        target_text = self.data.iloc[idx][self.target_lang]

        input_text = f"translate {self.source_lang} to {self.target_lang}: {input_text}"

        input_encoding = self.tokenizer(
            input_text, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt"
        )
        target_encoding = self.tokenizer(
            target_text, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt"
        )

        return {
            "input_ids": input_encoding["input_ids"].squeeze(0),  
            "attention_mask": input_encoding["attention_mask"].squeeze(0),  
            "labels": target_encoding["input_ids"].squeeze(0)  
        }

    def clean_target_tokens(self, target_ids):
        decoded = self.tokenizer.decode(target_ids, skip_special_tokens=True)
        cleaned = decoded.replace("Hindi:", "").strip()
        return self.tokenizer(cleaned, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")["input_ids"]


model_name = "t5-small"  
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

arr = ["Hindi", "Marathi",	"Tamil", "Kannada",	"Malayalam", "Afrikaans",	"Spanish",	"Chinese"]

for i in range(8):
    source_language = "English"  
    target_language = arr[i]    

    train_dataset = TranslationDataset(
        filepath='data.csv', 
        tokenizer=tokenizer, 
        source_lang=source_language, 
        target_lang=target_language
    )
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

    epochs = 10
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()
            
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()

            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(train_loader):.4f}")

    model.save_pretrained(f"fine_tuned_{model_name}")
    tokenizer.save_pretrained(f"fine_tuned_{model_name}")

print(f"Model fine-tuned and saved to 'fine_tuned_{model_name}'")


Epoch 1/10, Loss: 10.5763
Epoch 2/10, Loss: 8.2292
Epoch 3/10, Loss: 5.4282
Epoch 4/10, Loss: 4.4011
Epoch 5/10, Loss: 2.6057
Epoch 6/10, Loss: 2.6145
Epoch 7/10, Loss: 1.7584
Epoch 8/10, Loss: 1.4687
Epoch 9/10, Loss: 1.1537
Epoch 10/10, Loss: 1.0723
Epoch 1/10, Loss: 1.6335
Epoch 2/10, Loss: 1.1998
Epoch 3/10, Loss: 0.9290
Epoch 4/10, Loss: 0.8722
Epoch 5/10, Loss: 0.7659
Epoch 6/10, Loss: 0.6686
Epoch 7/10, Loss: 0.5387
Epoch 8/10, Loss: 0.5303
Epoch 9/10, Loss: 0.3575
Epoch 10/10, Loss: 0.3261
Epoch 1/10, Loss: 0.3651
Epoch 2/10, Loss: 0.3686
Epoch 3/10, Loss: 0.2642
Epoch 4/10, Loss: 0.2415
Epoch 5/10, Loss: 0.1777
Epoch 6/10, Loss: 0.1877
Epoch 7/10, Loss: 0.1984
Epoch 8/10, Loss: 0.1681
Epoch 9/10, Loss: 0.1687
Epoch 10/10, Loss: 0.1467
Epoch 1/10, Loss: 0.1431
Epoch 2/10, Loss: 0.1340
Epoch 3/10, Loss: 0.1265
Epoch 4/10, Loss: 0.1103
Epoch 5/10, Loss: 0.1006
Epoch 6/10, Loss: 0.0890
Epoch 7/10, Loss: 0.0885
Epoch 8/10, Loss: 0.0790
Epoch 9/10, Loss: 0.0925
Epoch 10/10, Loss: 0.

In [16]:
r = sr.Recognizer()
r.pause_threshold = 0.5  
r.energy_threshold = 300

main = 1


with sr.Microphone() as source:
  r.adjust_for_ambient_noise(source)

  try:
    t2 = "Welcome, to Real Time Speech Translation Application. \n Speaker one please tell me your language"
    print(t2)

    tts = gTTS(text=t2, lang='en')
    filename = "temp_audio.mp3"
    tts.save(filename)
    playsound.playsound(filename)
    os.remove(filename)
    model = Translator()

    l1 = r.listen(source)

    lang1 = r.recognize_google(l1, language='en')
    print("Speaker one selected:",lang1)
    lang1 = lang1[0:2].lower()

    t3 = "Speaker two please tell me your language"

    tts = gTTS(text=t3, lang='en')
    filename = "temp_audio.mp3"
    tts.save(filename)
    playsound.playsound(filename)
    os.remove(filename)

    l2 = r.listen(source)

    lang2 = r.recognize_google(l2, language='en')
    print("Speaker two selected:",lang2)
    lang2 = lang2[0:2].lower()
    
    if lang1 == 'ma':
      lang1 = 'mr'
    elif lang2 == 'ma':
      lang2 = 'mr'

    if lang1 == 'sp':
      lang1 = 'es'
    elif lang2 == 'sp':
      lang2 = 'es'

    t4 = "Speaker one goes first. \n Remember, one sentence at a time."

    tts = gTTS(text=t4, lang='en')
    filename = "temp_audio.mp3"
    tts.save(filename)
    playsound.playsound(filename)
    os.remove(filename)

    print("Listening....")

  except sr.UnknownValueError:
    print("Couldn't understand you")

  except sr.RequestError as e:
    print(f"Error: {e}")


  while True:
    if main % 4 == 0 :
      ch = input("Do you want to stop ? (Y or N)")
      if ch == "Y" or ch == "y":
        t1 = "Thank you, have a nice day"

        tran = model.translate(t1, src='en', dest=lang1)
        print(tran.text)
        tts = gTTS(text=tran.text, lang=lang1)
        filename = "temp_audio.mp3"
        tts.save(filename)
        playsound.playsound(filename)
        os.remove(filename)

        tran = model.translate(t1, src='en', dest=lang2)
        print(tran.text)
        tts = gTTS(text=tran.text, lang=lang2)
        filename = "temp_audio.mp3"
        tts.save(filename)
        playsound.playsound(filename)
        os.remove(filename)

        break

    try:
      audio1 = r.listen(source)
      text = r.recognize_google(audio1, language=lang1)
      print("Speaker One:", text)
      tran = model.translate(text, src=lang1, dest=lang2)
      print("Speaker One:",tran.text)

      tts = gTTS(text=tran.text, lang=lang2)
      filename = "temp_audio.mp3"
      tts.save(filename)
      playsound.playsound(filename)
      os.remove(filename)

      audio2 = r.listen(source)
      text2 = r.recognize_google(audio2, language=lang2)
      print("Speaker Two:",text2)
      tran2 = model.translate(text2, src=lang2, dest=lang1)
      print("Speaker Two:",tran2.text)

      tts = gTTS(text=tran2.text, lang=lang1)
      filename = "temp_audio.mp3"
      tts.save(filename)
      playsound.playsound(filename)
      os.remove(filename)

      main += 1

    except sr.UnknownValueError:
      print("Couldn't understand you")

    except sr.RequestError as e:
      print(f"Error: {e}")

Welcome, to Real Time Speech Translation Application. 
 Speaker one please tell me your language
Speaker one selected: Hindi
Speaker two selected: English
Listening....
Speaker One: आप कैसे हो
Speaker One: How are you
Speaker Two: I'm good
Speaker Two: मैं अच्छा हूँ
Speaker One: मैंने एक आदमी देखा
Speaker One: I saw a man
Speaker Two: how was he
Speaker Two: वह कैसा था
Couldn't understand you
Speaker One: वह क्रिकेट खेल रहा था
Speaker One: He was playing cricket
Speaker Two: how was he playing
Speaker Two: वह कैसे खेल रहा था
धन्यवाद आपका दिन अच्छा है
Thank you, have a nice day
